# Gradia.AI: Data preprocessing for HAM10000

Runs **once**, after `EDA.ipynb` and before training. It turns the raw download into one clean, fixed dataset that every training experiment reads, so image-only and metadata-fusion runs are compared on exactly the same data and split.

**What it does, in order**

1. Finds every image across the download folders and checks each one opens.
2. Cleans the metadata: consistent text, `age = 0` treated as missing, rare site `acral` merged into `foot`, `unknown` kept as a real category.
3. Fixes the label order (alphabetical, the same as `inference.py`).
4. Splits by **lesion**, about 70 / 15 / 15, stratified by class, so no lesion appears in two splits.
5. Computes statistics (median age, class weights) from the **training split only**, then fills missing ages.
6. Optionally saves every image resized to 341 × 256, the size the model resizes to anyway, so training loads faster.

**Outputs** (in `OUT_DIR`, outside the repo):

| File | What it is |
|---|---|
| `ham10000_clean.csv` | One row per image: label, split, cleaned metadata |
| `classes.json` | Index → class code, read by training and by `gradia-backend/inference.py` |
| `metadata_vocab.json` | Allowed sex and site values, plus training-set age statistics |
| `preprocess_report.json` | Everything that was changed, and class counts per split |
| `images_256/` | Resized images (if `RESIZE_IMAGES = True`) |

## 0. Setup

In [1]:
import os, json, random, shutil
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    IMAGE_DIRS = ["/content/drive/MyDrive/HAM10000"]
    CSV_PATH   = "/content/drive/MyDrive/HAM10000/HAM10000_metadata.csv"
    OUT_DIR    = "/content/drive/MyDrive/gradia_data"
else:
    # ▼ Local paths (Windows). Same as EDA.ipynb.
    IMAGE_DIRS = [
        r"C:\Users\krish.LAPTOP-3EPNQNP7\Downloads\HAM10000_images_part_1",
        r"C:\Users\krish.LAPTOP-3EPNQNP7\Downloads\HAM10000_images_part_2",
        r"C:\Users\krish.LAPTOP-3EPNQNP7\Documents\HAM10000",
    ]
    CSV_PATH = r"C:\Users\krish.LAPTOP-3EPNQNP7\Downloads\HAM10000_metadata.csv"
    OUT_DIR  = r"C:\Users\krish.LAPTOP-3EPNQNP7\Documents\gradia_data"   # outside the repo on purpose

    if os.environ.get("GRADIA_IMAGE_DIRS"):
        IMAGE_DIRS = os.environ["GRADIA_IMAGE_DIRS"].split(os.pathsep)
    CSV_PATH = os.environ.get("GRADIA_CSV", CSV_PATH)
    OUT_DIR  = os.environ.get("GRADIA_OUT", OUT_DIR)

# ── Options ──────────────────────────────────────────────────────────────────
SEED          = 42
TEST_FRAC     = 0.15     # held out until the very end
VAL_FRAC      = 0.15     # used to pick the best epoch and to calibrate
RESIZE_IMAGES = True     # save images at 256 px on the short side (341×256 for 600×450 originals)
RESIZE_SHORT  = 256
JPEG_QUALITY  = 95

random.seed(SEED); np.random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
REPORT = {"seed": SEED, "test_frac": TEST_FRAC, "val_frac": VAL_FRAC, "steps": []}
def log(step, **info):
    REPORT["steps"].append({"step": step, **info})
    print(f"✔ {step}", "" if not info else json.dumps(info, default=str))

print("Metadata CSV:", CSV_PATH, "| exists:", os.path.exists(CSV_PATH))
print("Output folder:", OUT_DIR)

Metadata CSV: C:\Users\krish.LAPTOP-3EPNQNP7\Downloads\HAM10000_metadata.csv | exists: True
Output folder: C:\Users\krish.LAPTOP-3EPNQNP7\Documents\gradia_data


## 1. Find and verify every image

In [2]:
IMAGE_PATHS = {}
for d in IMAGE_DIRS:
    if not os.path.isdir(d):
        print(f"WARNING: folder not found, skipping: {d}")
        continue
    n = 0
    for f in os.listdir(d):
        if f.lower().endswith(".jpg"):
            IMAGE_PATHS.setdefault(f[:-4], os.path.join(d, f))
            n += 1
    print(f"{n:>6} .jpg files in {d}")
print("Unique images found:", len(IMAGE_PATHS))

raw = pd.read_csv(CSV_PATH)
print("Metadata rows:", len(raw))

def verify(image_id):
    path = IMAGE_PATHS.get(image_id)
    if path is None:
        return image_id, "missing file"
    try:
        with Image.open(path) as im:
            im.verify()               # checks the file isn't truncated or corrupt
        return image_id, ""
    except Exception as e:
        return image_id, f"unreadable: {e}"

with ThreadPoolExecutor(max_workers=16) as ex:
    results = list(tqdm(ex.map(verify, raw["image_id"]), total=len(raw), desc="Verifying images"))
problems = {iid: err for iid, err in results if err}
print("Images with problems:", len(problems))
for iid, err in list(problems.items())[:10]:
    print("  ", iid, err)
log("verify_images", found=len(IMAGE_PATHS), problems=len(problems))

  5000 .jpg files in C:\Users\krish.LAPTOP-3EPNQNP7\Downloads\HAM10000_images_part_1
  5015 .jpg files in C:\Users\krish.LAPTOP-3EPNQNP7\Downloads\HAM10000_images_part_2
 10015 .jpg files in C:\Users\krish.LAPTOP-3EPNQNP7\Documents\HAM10000
Unique images found: 10015
Metadata rows: 10015


Verifying images:   0%|          | 0/10015 [00:00<?, ?it/s]

Images with problems: 0
✔ verify_images {"found": 10015, "problems": 0}


## 2. Clean the metadata

Nothing is deleted unless it is genuinely broken (an image that is missing or won't open). Unknown values stay as their own category, because "not recorded" is real information.

In [3]:
df = raw.copy()
n0 = len(df)

# Rows we cannot train on: image missing or corrupt
df = df[~df["image_id"].isin(problems)]
dropped_broken = n0 - len(df)

# Exact duplicate rows and duplicate image_ids (EDA found none; this keeps the notebook safe on other copies)
dup_rows = int(df.duplicated().sum());               df = df.drop_duplicates()
dup_ids  = int(df["image_id"].duplicated().sum());   df = df.drop_duplicates("image_id", keep="first")

# Consistent text: lowercase, trimmed
for col in ["dx", "dx_type", "sex", "localization"]:
    df[col] = df[col].astype(str).str.strip().str.lower()

# Sex: anything other than male/female becomes "unknown"
sex_before = df["sex"].value_counts().to_dict()
df.loc[~df["sex"].isin(["male", "female"]), "sex"] = "unknown"

# Body site: missing -> unknown; merge "acral" (palms/soles, only 7 images) into "foot"
df["localization"] = df["localization"].replace({"nan": "unknown", "": "unknown"})
n_acral = int((df["localization"] == "acral").sum())
df["localization"] = df["localization"].replace({"acral": "foot"})

# Age: NaN or 0 means "not recorded"
n_age_nan  = int(df["age"].isna().sum())
n_age_zero = int((df["age"] == 0).sum())
df["age_missing"] = (df["age"].isna() | (df["age"] == 0)).astype(int)
df.loc[df["age_missing"] == 1, "age"] = np.nan

# A lesion must have exactly one diagnosis
conflicts = df.groupby("lesion_id")["dx"].nunique()
assert (conflicts == 1).all(), f"{(conflicts > 1).sum()} lesions have conflicting labels"

log("clean_metadata", rows_in=n0, dropped_broken=dropped_broken, duplicate_rows=dup_rows,
    duplicate_image_ids=dup_ids, sex_values_before=sex_before, acral_merged_into_foot=n_acral,
    age_nan=n_age_nan, age_zero_treated_missing=n_age_zero, rows_out=len(df))

print("\nSex:", df["sex"].value_counts().to_dict())
print("Sites:", df["localization"].value_counts().to_dict())
print("Age missing:", int(df["age_missing"].sum()))

✔ clean_metadata {"rows_in": 10015, "dropped_broken": 0, "duplicate_rows": 0, "duplicate_image_ids": 0, "sex_values_before": {"male": 5406, "female": 4552, "unknown": 57}, "acral_merged_into_foot": 7, "age_nan": 57, "age_zero_treated_missing": 39, "rows_out": 10015}

Sex: {'male': 5406, 'female': 4552, 'unknown': 57}
Sites: {'back': 2192, 'lower extremity': 2077, 'trunk': 1404, 'upper extremity': 1118, 'abdomen': 1022, 'face': 745, 'chest': 407, 'foot': 326, 'unknown': 234, 'neck': 168, 'scalp': 128, 'hand': 90, 'ear': 56, 'genital': 48}
Age missing: 96


## 3. Fix the label order

Alphabetical, exactly like scikit-learn's `LabelEncoder` and `DEFAULT_CLASS_ORDER` in `gradia-backend/inference.py`. Saved to `classes.json`, so training and the app read the same file and can never disagree.

In [4]:
CLASSES = sorted(df["dx"].unique())                 # ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
assert len(CLASSES) == 7, CLASSES
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
df["label"] = df["dx"].map(CLASS_TO_IDX).astype(int)

classes_upper = [c.upper() for c in CLASSES]
assert classes_upper == ["AKIEC", "BCC", "BKL", "DF", "MEL", "NV", "VASC"], "order differs from inference.py"
with open(f"{OUT_DIR}/classes.json", "w") as f:
    json.dump(classes_upper, f)
print({i: c for i, c in enumerate(classes_upper)})
log("encode_labels", classes=classes_upper)

{0: 'AKIEC', 1: 'BCC', 2: 'BKL', 3: 'DF', 4: 'MEL', 5: 'NV', 6: 'VASC'}
✔ encode_labels {"classes": ["AKIEC", "BCC", "BKL", "DF", "MEL", "NV", "VASC"]}


## 4. Split by lesion

Every photo of a lesion goes to the same split. The EDA showed the old image-level split leaked 39% of validation images.

`StratifiedGroupKFold` keeps whole lesions together while keeping class proportions similar. Test is carved out first, then validation from the rest.

In [5]:
def grouped_holdout(frame, frac, seed):
    k = max(2, round(1 / frac))
    sgkf = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
    rest_idx, hold_idx = next(sgkf.split(frame, frame["label"], groups=frame["lesion_id"]))
    return frame.iloc[rest_idx], frame.iloc[hold_idx]

rest, test = grouped_holdout(df, TEST_FRAC, SEED)
val_share_of_rest = VAL_FRAC / (1 - TEST_FRAC)
train, val = grouped_holdout(rest, val_share_of_rest, SEED)

df["split"] = "train"
df.loc[df["image_id"].isin(val["image_id"]), "split"] = "val"
df.loc[df["image_id"].isin(test["image_id"]), "split"] = "test"

# Checks: no lesion in two splits, every class in every split
lesion_splits = df.groupby("lesion_id")["split"].nunique()
assert (lesion_splits == 1).all(), "a lesion appears in more than one split"
per_split = pd.crosstab(df["dx"], df["split"])[["train", "val", "test"]]
assert (per_split > 0).all().all(), "some class is missing from a split"

table = per_split.copy()
table["total"] = table.sum(axis=1)
for s in ["train", "val", "test"]:
    table[f"{s} %"] = (per_split[s] / table["total"] * 100).round(1)
display(table)
sizes = df["split"].value_counts()[["train", "val", "test"]]
print("Images per split:", sizes.to_dict(), "| shares:", (sizes / len(df) * 100).round(1).to_dict())
print("Lesions per split:", df.groupby("split")["lesion_id"].nunique()[["train", "val", "test"]].to_dict())
log("split_by_lesion", images=sizes.to_dict(), per_class=per_split.to_dict(orient="index"))

split,train,val,test,total,train %,val %,test %
dx,,,,,,,
akiec,234,47,46,327,71.6,14.4,14.1
bcc,366,74,74,514,71.2,14.4,14.4
bkl,785,157,157,1099,71.4,14.3,14.3
df,82,16,17,115,71.3,13.9,14.8
mel,795,159,159,1113,71.4,14.3,14.3
nv,4789,958,958,6705,71.4,14.3,14.3
vasc,102,20,20,142,71.8,14.1,14.1


Images per split: {'train': 7153, 'val': 1431, 'test': 1431} | shares: {'train': 71.4, 'val': 14.3, 'test': 14.3}
Lesions per split: {'train': 5336, 'val': 1067, 'test': 1067}
✔ split_by_lesion {"images": {"train": 7153, "val": 1431, "test": 1431}, "per_class": {"akiec": {"train": 234, "val": 47, "test": 46}, "bcc": {"train": 366, "val": 74, "test": 74}, "bkl": {"train": 785, "val": 157, "test": 157}, "df": {"train": 82, "val": 16, "test": 17}, "mel": {"train": 795, "val": 159, "test": 159}, "nv": {"train": 4789, "val": 958, "test": 958}, "vasc": {"train": 102, "val": 20, "test": 20}}}


## 5. Statistics from the training split only

The median age and the class weights are computed on **train only**. Computing them on all data would let information from validation and test leak into training.

In [6]:
tr = df[df["split"] == "train"]
AGE_MEDIAN = float(tr["age"].median())
AGE_MEAN, AGE_STD = float(tr["age"].mean()), float(tr["age"].std())
df["age_filled"] = df["age"].fillna(AGE_MEDIAN)

train_counts = tr["label"].value_counts().sort_index()
w = 1.0 / train_counts
class_weights = (w / w.mean()).round(4).tolist()          # averaged to 1, in CLASSES order

SEXES = ["female", "male", "unknown"]
SITES = sorted(df["localization"].unique())
vocab = {
    "sex": SEXES,
    "localization": SITES,
    "age_median_train": AGE_MEDIAN, "age_mean_train": round(AGE_MEAN, 3), "age_std_train": round(AGE_STD, 3),
    "class_weights_train": dict(zip(classes_upper, class_weights)),
    "note": "Use these exact lists, in this order, to one-hot encode sex and site in training AND inference.",
}
with open(f"{OUT_DIR}/metadata_vocab.json", "w") as f:
    json.dump(vocab, f, indent=2)

print(f"Training median age: {AGE_MEDIAN}  (filled {int(df['age_missing'].sum())} missing ages)")
print("Sites:", SITES)
print("Class weights (train):", dict(zip(classes_upper, class_weights)))
log("train_only_stats", age_median=AGE_MEDIAN, sites=SITES, class_weights=dict(zip(classes_upper, class_weights)))

Training median age: 50.0  (filled 96 missing ages)
Sites: ['abdomen', 'back', 'chest', 'ear', 'face', 'foot', 'genital', 'hand', 'lower extremity', 'neck', 'scalp', 'trunk', 'unknown', 'upper extremity']
Class weights (train): {'AKIEC': 0.9423, 'BCC': 0.6025, 'BKL': 0.2809, 'DF': 2.6891, 'MEL': 0.2774, 'NV': 0.046, 'VASC': 2.1618}
✔ train_only_stats {"age_median": 50.0, "sites": ["abdomen", "back", "chest", "ear", "face", "foot", "genital", "hand", "lower extremity", "neck", "scalp", "trunk", "unknown", "upper extremity"], "class_weights": {"AKIEC": 0.9423, "BCC": 0.6025, "BKL": 0.2809, "DF": 2.6891, "MEL": 0.2774, "NV": 0.046, "VASC": 2.1618}}


## 6. Optional: save resized images

Training resizes every image to 256 px on the short side anyway. Doing it once here means training reads 341 × 256 files instead of 600 × 450, which is much faster, especially from Google Drive. Already-resized files are skipped, so re-running is cheap.

In [7]:
RESIZED_DIR = f"{OUT_DIR}/images_{RESIZE_SHORT}"

def resize_one(image_id):
    dst = f"{RESIZED_DIR}/{image_id}.jpg"
    if os.path.exists(dst):
        return ""
    try:
        with Image.open(IMAGE_PATHS[image_id]) as im:
            im = im.convert("RGB")
            w, h = im.size
            s = RESIZE_SHORT / min(w, h)
            im = im.resize((round(w * s), round(h * s)), Image.BICUBIC)
            im.save(dst, quality=JPEG_QUALITY)
        return ""
    except Exception as e:
        return f"{image_id}: {e}"

if RESIZE_IMAGES:
    os.makedirs(RESIZED_DIR, exist_ok=True)
    with ThreadPoolExecutor(max_workers=16) as ex:
        errs = [e for e in tqdm(ex.map(resize_one, df["image_id"]), total=len(df), desc="Resizing") if e]
    print("Resize errors:", len(errs), errs[:5])
    with Image.open(f"{RESIZED_DIR}/{df['image_id'].iloc[0]}.jpg") as im:
        print("Example resized size:", im.size)
    log("resize_images", folder=RESIZED_DIR, short_side=RESIZE_SHORT, errors=len(errs))
else:
    print("Skipped (RESIZE_IMAGES = False)")

Resizing:   0%|          | 0/10015 [00:00<?, ?it/s]

Resize errors: 0 []
Example resized size: (341, 256)
✔ resize_images {"folder": "C:\\Users\\krish.LAPTOP-3EPNQNP7\\Documents\\gradia_data/images_256", "short_side": 256, "errors": 0}


## 7. Save the clean dataset

Paths are **not** stored, because they differ between your laptop and Colab. The training notebook rebuilds them from `image_id` and its own image folder.

In [8]:
clean = df[["image_id", "lesion_id", "dx", "label", "split",
            "age", "age_filled", "age_missing", "sex", "localization", "dx_type"]].sort_values(["split", "image_id"])
clean.to_csv(f"{OUT_DIR}/ham10000_clean.csv", index=False)

REPORT["rows"] = int(len(clean))
REPORT["classes"] = classes_upper
REPORT["per_split"] = per_split.to_dict(orient="index")
REPORT["feature_note"] = "dx_type is kept for analysis only. Never use it as a model input (label leakage)."
with open(f"{OUT_DIR}/preprocess_report.json", "w") as f:
    json.dump(REPORT, f, indent=2, default=str)

display(clean.head())
print("\nSaved to", OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    print("  ", f)

,image_id,lesion_id,dx,label,split,age,age_filled,age_missing,sex,localization,dx_type
700,ISIC_0024312,HAM_0001359,bkl,2,test,75.0,75.0,0,male,lower extremity,histo
8123,ISIC_0024319,HAM_0003062,nv,5,test,55.0,55.0,0,male,lower extremity,histo
9931,ISIC_0024329,HAM_0002954,akiec,0,test,75.0,75.0,0,female,lower extremity,histo
8336,ISIC_0024349,HAM_0000025,nv,5,test,35.0,35.0,0,male,lower extremity,histo
6132,ISIC_0024350,HAM_0006264,nv,5,test,35.0,35.0,0,male,lower extremity,follow_up



Saved to C:\Users\krish.LAPTOP-3EPNQNP7\Documents\gradia_data
   classes.json
   ham10000_clean.csv
   images_256
   metadata_vocab.json
   preprocess_report.json


## 8. Final checks

In [9]:
check = pd.read_csv(f"{OUT_DIR}/ham10000_clean.csv")
cls = json.load(open(f"{OUT_DIR}/classes.json"))

assert check["image_id"].is_unique
assert check.groupby("lesion_id")["split"].nunique().max() == 1
assert check["age_filled"].notna().all()
assert set(check["sex"]) <= set(vocab["sex"])
assert set(check["localization"]) <= set(vocab["localization"])
assert (check["dx"].str.upper().map({c: i for i, c in enumerate(cls)}) == check["label"]).all()
if RESIZE_IMAGES:
    assert all(os.path.exists(f"{RESIZED_DIR}/{i}.jpg") for i in check["image_id"])

print("All checks passed ✔")
print(f"{len(check)} images | {check['lesion_id'].nunique()} lesions | splits: {check['split'].value_counts().to_dict()}")

All checks passed ✔
10015 images | 7470 lesions | splits: {'train': 7153, 'test': 1431, 'val': 1431}
